# Debug One Benchmark Question

Use this notebook to run just one benchmark item or one custom question through the same pipeline as `eval_nurse_benchmark.py`.

It is useful when you want to check whether a fix worked without rerunning the full benchmark.

In [207]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path
from pprint import pprint

from elasticsearch import ElasticsearchWarning

warnings.filterwarnings("ignore", category=ElasticsearchWarning)


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "02_configs").exists() and (candidate / "04_src").exists():
            return candidate
    raise RuntimeError("Could not find repo root from current working directory")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "04_src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from eval_nurse_benchmark import compare_answer, run_pipeline
from historyAssistant.app.runtime import load_runtime
from historyAssistant.execution import open_es_connection

BENCHMARK_PATH = REPO_ROOT / "00_data" / "historyCalls" / "04_ground_truth" / "ground_truth_benchmark_nurse_fast_30.json"
benchmark_records = json.loads(BENCHMARK_PATH.read_text(encoding="utf-8"))
benchmark_by_id = {item["id"]: item for item in benchmark_records}

bundle, client, examples = load_runtime()
es = open_es_connection(config_bundle=bundle)

print("Repo root:", REPO_ROOT)
print("Benchmark items:", len(benchmark_records))

Repo root: C:\Users\S_YZH\OneDrive - Televic Group NV\Desktop\HistoryAssistant
Benchmark items: 30


In [206]:
# Pick one mode:
# 1. BENCHMARK_ID = "T040"
# 2. CUSTOM_QUESTION = "Your question here"

BENCHMARK_ID = "T083"
CUSTOM_QUESTION = None

assert BENCHMARK_ID or CUSTOM_QUESTION, "Set BENCHMARK_ID or CUSTOM_QUESTION"

if BENCHMARK_ID:
    gt = benchmark_by_id[BENCHMARK_ID]
    question = gt["question"]
    expected_scenario = gt["scenario"]
else:
    gt = None
    question = CUSTOM_QUESTION
    expected_scenario = "simple_lookup"

print("Question:")
print(question)
print()
print("Expected scenario:", expected_scenario)
if gt:
    print("Expected output type:", gt.get("expected_output_type"))
    print("Expected answer:")
    pprint(gt.get("expected_answer"), sort_dicts=False)

Question:
Compare calls from ward 1V ZONE 1A at week of 2024-04-10 to ward 1V ZONE 1B at week of 2024-04-24

Expected scenario: filtering_ranking
Expected output type: scalar
Expected answer:
'1V ZONE 1A / week of 10/04/2024'


In [204]:
result = run_pipeline(question, expected_scenario, bundle, client, examples, es)

match_ok = None
match_detail = "custom question: no benchmark comparison"
if gt is not None:
    match_ok, match_detail = compare_answer(result, gt)

print("Match:", match_ok)
print("Match detail:", match_detail)
print("Scenario predicted:", result.get("scenario"))
print("Scenario expected:", expected_scenario)
print("Scenario match:", result.get("scenario_match"))
print("Total time (sec):", result.get("total_time_sec"))

Match: False
Match detail: scalar got=1V ZONE 1A want=1V ZONE 1A / week of 10/04/2024
Scenario predicted: filtering_ranking
Scenario expected: filtering_ranking
Scenario match: True
Total time (sec): 9.657


In [119]:
compact = {
    "scenario": result.get("scenario"),
    "scenario_match": result.get("scenario_match"),
    "validation_valid": result.get("validation_valid"),
    "repairs": result.get("repairs"),
    "timings": result.get("timings"),
    "scalar_answer": result.get("scalar_answer"),
    "record_count": len(result.get("records") or []),
    "where_debug": result.get("where_debug"),
    "comparison_summary": result.get("comparison_summary"),
    "error": result.get("error"),
}
pprint(compact, sort_dicts=False)

{'scenario': 'filtering_ranking',
 'scenario_match': True,
 'validation_valid': True,
 'repairs': ['Filled missing aggregation from metric catalog',
             'Filled missing scope_policy from metric catalog',
             'Filled missing outlier_policy from metric catalog'],
 'timings': {'guardrails_sec': 0.047,
             'normalization_sec': 0.0,
             'routing_sec': 2.234,
             'retrieval_sec': 0.094,
             'planning_hints_sec': 0.0,
             'planning_sec': 3.797,
             'execution_sec': 0.125},
 'scalar_answer': 'week of 03/06/2024',
 'record_count': 1,
 'where_debug': ['call_date between 2024-06-03 and 2024-06-09',
                 'call_date between 2024-06-03 and 2024-06-09'],
 'comparison_summary': {'main_label': 'week of 03/06/2024',
                        'comparison_label': 'week of 10/06/2024',
                        'winner_label': 'week of 03/06/2024',
                        'preferred_direction': 'desc',
                        '

In [120]:
records = result.get("records") or []
print("Preview rows:", len(records))
for row in records[:10]:
    pprint(row, sort_dicts=False)
    print("-" * 60)

Preview rows: 1
{'metric_value': 1364}
------------------------------------------------------------


In [121]:
planning_entry = result.get("planning_entry") or {}
validation = planning_entry.get("validation") or {}
canonical_plan = validation.get("canonical_plan") or result.get("canonical_plan")
print("Canonical plan:")
pprint(canonical_plan, sort_dicts=False)

Canonical plan:
{'request_type': 'qa',
 'target_tables': ['historyCalls'],
 'normalized_sub_request': 'Between the week of 03/06/2024 and the week of '
                           '10/06/2024, which week had more calls?',
 'sub_request_id': 1,
 'classification_confidence': None,
 'filters': [{'field': 'call_date',
              'op': 'between',
              'value': ['2024-06-03', '2024-06-09']}],
 'metric': 'total_calls',
 'group_by': [],
 'aggregation': 'count',
 'ranking_order': None,
 'display_order': [],
 'order_by': None,
 'limit': None,
 'scope_policy': 'raw',
 'outlier_policy': 'none',
 'date_rule': 'custom_range',
 'date_params': None,
 'date_range': {'start_date': '2024-06-03', 'end_date': '2024-06-09'},
 'compare_to': {'date_rule': 'custom_range',
                'date_range': {'start_date': '2024-06-10',
                               'end_date': '2024-06-16'},
                'filters': [],
                'date_params': None}}
